In [ ]:
from google.colab import userdata
from huggingface_hub import login
import sys
import os

os.environ['GITHUB_TOKEN'] = userdata.get('GITHUB_TOKEN')
!git clone https://$GITHUB_TOKEN@github.com/Constantine1824/C10-team-comoe.git
sys.path.append('/content/C10-team-comoe')
login(token=userdata.get('HF_TOKEN'))
%cd /content/C10-team-comoe/scripts
#!git pull origin main

In [ ]:
import importlib
importlib.invalidate_caches()

import importlib.util
print(importlib.util.find_spec('finetune'))

In [ ]:
!pip install trl peft bitsandbytes
import numpy as np
import pandas as pd
from finetune.trainer import collate_fn, finetune, evaluate, prepare_data
from format import format_train_data, format_test_data

In [ ]:
data = pd.read_csv('../data/train_qa.csv')
data.head()

In [ ]:
doc = pd.read_csv('../data/documents.csv')
doc.head()

In [ ]:
data = data.merge(doc[['document_id', 'text']], on='document_id', how='left')
data = data.rename(columns={'text':'context'})
data.head()

In [ ]:
from datasets import Dataset
from rag.retriever import HybridRetriever

retriever = HybridRetriever()
test_data = pd.read_csv('../data/test_questions.csv')
train_data, test_data, _ = prepare_data(train_data, test_data, retriever)

In [ ]:
trainer = finetune(train_data)
eval = evaluate(trainer, test_data, batch_size=11)
eval

In [ ]:
test_df = pd.read_csv('data/test_questions.csv')
test_df.head()

In [ ]:
qid = test_df['QuestionId']
qid

In [ ]:
def save_submission(result):
    qid = test_df['QuestionId']
    cleaned = pd.Series([
        s.lstrip('\n').removeprefix('Answer:').strip()
        for s in result
    ])
    submission = qid.to_frame()
    submission['Answer'] = cleaned
    assert list(submission.columns) == ['QuestionId', 'Answer']
    assert len(submission) == len(test_df) == 11
    assert submission['QuestionId'].tolist() == test_df['QuestionId'].tolist()
    assert submission['Answer'].str.strip().ne('').all()
    submission.to_csv('/kaggle/working/submission.csv', index=False)
    print(submission)
    print('Saved /kaggle/working/submission.csv')

save_submission(eval)